# Parte 3 — Ecuaciones elípticas
## 3.2 Propiedades de las funciones armónicas
### 3.2.01 Representación por la solución fundamental, propiedad del promedio y subarmonicidad

**Fuente principal:** página 6 de las notas manuscritas del archivo `Ecuación De Onda.pdf`.

Este notebook continúa exactamente después de `03.1.01_Aplicaciones_y_solucion_fundamental`.
No se avanza todavía a la página 7 de las notas.

## Convenciones editoriales

- El bloque **Transcripción de las notas** conserva el orden y la notación manuscrita.
- Las hipótesis que no están escritas, pero son necesarias, aparecen como **Aclaración**.
- Los argumentos completados aparecen como **Demostración añadida**.
- Las correcciones de signo o regularidad aparecen como **Corrección editorial**.
- Toda la matemática en Markdown usa únicamente `$...$` y `$$...$$`.

> **Versión corregida para renderizado matemático.**  
> Todas las expresiones usan únicamente `$...$` y `$$...$$`, compatibles con
> Jupyter Notebook, JupyterLab y VS Code. También se sustituyó `\fint` por
> cocientes explícitos de integrales.


# Simulaciones y gráficas

Las celdas de esta sección se ejecutan directamente. No se requiere cambiar ninguna bandera como
`VIDEO=True`.

Se incluyen tres visualizaciones:

1. comparación de promedios sobre circunferencias y discos;
2. verificación numérica de la propiedad del promedio para una función armónica;
3. comparación con una función subarmónica y otra no armónica.

La GPU se utiliza automáticamente mediante CuPy cuando está disponible y el tamaño de la malla
lo justifica.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False
try:
    import cupy as cp
    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend: CuPy/CUDA")
    else:
        xp = np
        print("Backend: NumPy/CPU")
except Exception as exc:
    xp = np
    print("Backend: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)

def to_cpu(a):
    return cp.asnumpy(a) if GPU_AVAILABLE else np.asarray(a)

def circle_average(fun, center, radius, n_theta=20000):
    theta = xp.linspace(0.0, 2.0 * math.pi, n_theta, endpoint=False)
    x = center[0] + radius * xp.cos(theta)
    y = center[1] + radius * xp.sin(theta)
    values = fun(x, y)
    return float(to_cpu(xp.mean(values)))

def disk_average(fun, center, radius, n_r=1200, n_theta=2400):
    # Cuadratura polar: integral_B u = ∫_0^R ∫_0^{2π} u(r,θ) r dθ dr
    r = xp.linspace(0.0, radius, n_r)
    theta = xp.linspace(0.0, 2.0 * math.pi, n_theta, endpoint=False)
    rr, tt = xp.meshgrid(r, theta, indexing="ij")
    x = center[0] + rr * xp.cos(tt)
    y = center[1] + rr * xp.sin(tt)
    values = fun(x, y)
    radial_integrand = xp.mean(values, axis=1) * r
    integral = 2.0 * math.pi * xp.trapz(radial_integrand, r)
    return float(to_cpu(integral / (math.pi * radius**2)))

## Simulación 3.2.A — Propiedad del promedio para una función armónica

Consideramos

$$
u(x,y)=e^x\cos y.
$$

Como

$$
u_{xx}=e^x\cos y,
\qquad
u_{yy}=-e^x\cos y,
$$

se tiene

$$
\Delta u=0.
$$

Para varios centros y radios se comparan:

$$
u(x_0),
$$

$$
\frac{1}{|\partial B_r(x_0)|}
\int_{\partial B_r(x_0)}u\,dS,
$$

y

$$
\frac{1}{|B_r(x_0)|}
\int_{B_r(x_0)}u\,dx.
$$

In [ ]:
def u_harmonic(x, y):
    return xp.exp(x) * xp.cos(y)

centers = [(-0.4, 0.25), (0.0, 0.0), (0.55, -0.35)]
radii = np.linspace(0.08, 0.9, 18)

records = []
for center in centers:
    point_value = math.exp(center[0]) * math.cos(center[1])
    circle_vals = []
    disk_vals = []
    for r in radii:
        circle_vals.append(circle_average(u_harmonic, center, float(r)))
        disk_vals.append(disk_average(u_harmonic, center, float(r)))
    records.append((center, point_value, np.asarray(circle_vals), np.asarray(disk_vals)))

fig, ax = plt.subplots(figsize=(9, 6))
for center, point_value, circle_vals, disk_vals in records:
    label_center = rf"$x_0=({center[0]:.2f},{center[1]:.2f})$"
    ax.plot(radii, circle_vals - point_value, marker="o", ms=3,
            label=label_center + " — esfera")
    ax.plot(radii, disk_vals - point_value, linestyle="--",
            label=label_center + " — bola")

ax.axhline(0.0, linewidth=1.0)
ax.set_xlabel(r"$r$")
ax.set_ylabel("promedio menos valor en el centro")
ax.set_title("Verificación numérica de la propiedad del promedio")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
path = FIG_DIR / "03.2.A_propiedad_promedio_armonica.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

max_error = max(
    max(np.max(np.abs(c - p)), np.max(np.abs(d - p)))
    for _, p, c, d in records
)
print("Error máximo observado:", max_error)
print(path.resolve())

## Simulación 3.2.B — Comparación con funciones no armónicas

Se consideran

$$
u_1(x,y)=e^x\cos y,
$$

$$
u_2(x,y)=x^2+y^2,
$$

y

$$
u_3(x,y)=x^2-y^2+x^2+y^2.
$$

La primera es armónica. La segunda satisface

$$
\Delta u_2=4>0,
$$

por lo que es subarmónica. La tercera no es armónica y contiene una parte armónica y una parte
estrictamente subarmónica.

Para una función subarmónica suave se espera que el valor central no exceda sus promedios.

In [ ]:
def u_sub(x, y):
    return x**2 + y**2

def u_mixed(x, y):
    return (x**2 - y**2) + (x**2 + y**2)

center = (0.35, -0.15)
radii2 = np.linspace(0.04, 1.0, 24)

functions = [
    ("armónica", u_harmonic, math.exp(center[0]) * math.cos(center[1])),
    ("subarmónica", u_sub, center[0]**2 + center[1]**2),
    ("mixta", u_mixed, 2.0 * center[0]**2),
]

fig, ax = plt.subplots(figsize=(9, 6))
for name, fun, point_value in functions:
    means = np.asarray([disk_average(fun, center, float(r)) for r in radii2])
    ax.plot(radii2, means - point_value, marker="o", ms=3, label=name)

ax.axhline(0.0, linewidth=1.0)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\frac{1}{|B_r(x_0)|}\int_{B_r(x_0)}u-u(x_0)$")
ax.set_title("Promedios volumétricos: caso armónico y subarmónico")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
path = FIG_DIR / "03.2.B_comparacion_subarmonica.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

## Simulación 3.2.C — Barrido espacial del defecto de promedio

Para un radio fijo se calcula

$$
D_r u(x)
=
\frac{1}{|B_r(x)|}\int_{B_r(x)}u(y)\,dy-u(x)
$$

sobre una malla de centros.

- Si $u$ es armónica, el defecto debe ser aproximadamente cero.
- Si $u$ es subarmónica, el defecto debe ser no negativo.

In [ ]:
grid_n = 170 if GPU_AVAILABLE else 110
radius = 0.28
cx = np.linspace(-1.0, 1.0, grid_n)
cy = np.linspace(-1.0, 1.0, grid_n)

# Muestreo polar moderado para muchos centros.
n_r = 90 if GPU_AVAILABLE else 55
n_theta = 180 if GPU_AVAILABLE else 100
rr = xp.linspace(0.0, radius, n_r)
tt = xp.linspace(0.0, 2.0 * math.pi, n_theta, endpoint=False)
RR, TT = xp.meshgrid(rr, tt, indexing="ij")
DX = RR * xp.cos(TT)
DY = RR * xp.sin(TT)

def many_center_disk_defect(fun, point_fun):
    defects = np.empty((len(cy), len(cx)), dtype=float)
    for j, y0 in enumerate(cy):
        x0 = xp.asarray(cx)[:, None, None]
        yy0 = float(y0)
        vals = fun(x0 + DX[None, :, :], yy0 + DY[None, :, :])
        angle_mean = xp.mean(vals, axis=2)
        integrand = angle_mean * rr[None, :]
        integrals = 2.0 * math.pi * xp.trapz(integrand, rr, axis=1)
        means = integrals / (math.pi * radius**2)
        pvals = point_fun(xp.asarray(cx), yy0)
        defects[j, :] = to_cpu(means - pvals)
    return defects

def point_h(x, y):
    return xp.exp(x) * xp.cos(y)

def point_sub(x, y):
    return x**2 + y**2

defect_h = many_center_disk_defect(u_harmonic, point_h)
defect_s = many_center_disk_defect(u_sub, point_sub)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    defect_h,
    origin="lower",
    extent=[cx.min(), cx.max(), cy.min(), cy.max()],
    interpolation="nearest",
)
fig.colorbar(im, ax=ax, label="defecto de promedio")
ax.set_xlabel(r"$x_0$")
ax.set_ylabel(r"$y_0$")
ax.set_title("Defecto de promedio para una función armónica")
fig.tight_layout()
path1 = FIG_DIR / "03.2.C_defecto_armonico.png"
fig.savefig(path1, dpi=220)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    defect_s,
    origin="lower",
    extent=[cx.min(), cx.max(), cy.min(), cy.max()],
    interpolation="nearest",
)
fig.colorbar(im, ax=ax, label="defecto de promedio")
ax.set_xlabel(r"$x_0$")
ax.set_ylabel(r"$y_0$")
ax.set_title("Defecto de promedio para una función subarmónica")
fig.tight_layout()
path2 = FIG_DIR / "03.2.C_defecto_subarmonico.png"
fig.savefig(path2, dpi=220)
plt.show()
plt.close(fig)

print("Máximo error armónico:", np.max(np.abs(defect_h)))
print("Mínimo defecto subarmónico:", np.min(defect_s))
print(path1.resolve())
print(path2.resolve())

# 3.2.1 Representación mediante la solución fundamental

#### **Transcripción literal de las notas**

**Teorema.**

Sean $\Omega$ abierto y acotado y $f\in C^2(\Omega)$.

Si

$$
v(x)=\int_\Omega \Phi(x-y)f(y)\,dy,
$$

entonces

$$
-\Delta v=f
\qquad\text{en }\Omega.
$$

## **Teorema 3.2.1 (Potencial newtoniano de una fuente suave).**

Sea $\Omega\subset\mathbb R^n$ abierto y acotado, con $n\ge2$, y sea
$f\in C_c^2(\Omega)$. Extiéndase $f$ por cero fuera de $\Omega$ y defínase

$$
v(x)
=
\int_{\mathbb R^n}\Phi(x-y)f(y)\,dy
=
\int_\Omega\Phi(x-y)f(y)\,dy.
$$

Entonces $v$ es una solución distribucional de

$$
-\Delta v=f
\qquad\text{en }\mathbb R^n.
$$

Además, por regularidad interior, $v\in C^2(\Omega)$ y la ecuación se satisface puntualmente
en $\Omega$.

### **Aclaración.**

En las notas aparece $f\in C^2(\Omega)$. Para justificar directamente la extensión por cero sin
términos de frontera se introduce aquí la hipótesis $f\in C_c^2(\Omega)$. La versión para datos que
no se anulan cerca de $\partial\Omega$ requiere localizar el argumento o trabajar con extensiones
adecuadas.

### Demostración añadida

Sea $\varphi\in C_c^\infty(\mathbb R^n)$. Por Fubini y por la identidad
$-\Delta\Phi=\delta_0$,

$$
\begin{aligned}
\langle-\Delta v,\varphi\rangle
&=
-\int_{\mathbb R^n}v(x)\Delta\varphi(x)\,dx\\
&=
-\int_{\mathbb R^n}
\int_{\mathbb R^n}
\Phi(x-y)f(y)\Delta\varphi(x)\,dy\,dx\\
&=
\int_{\mathbb R^n}
f(y)
\left[
-\int_{\mathbb R^n}
\Phi(x-y)\Delta\varphi(x)\,dx
\right]dy\\
&=
\int_{\mathbb R^n}f(y)\varphi(y)\,dy.
\end{aligned}
$$

Por tanto,

$$
-\Delta v=f
$$

en el sentido de distribuciones. La regularidad interior de Poisson proporciona la afirmación
clásica dentro de $\Omega$.

$\square$

### Ejercicios — Sección 3.2.1

1. Sea $f\in C_c^\infty(\mathbb R^n)$. Demuestre que $\Phi*f$ está bien definida y es
   localmente integrable.

2. Justifique rigurosamente el intercambio del orden de integración en la demostración del
   Teorema 3.2.1.

3. Sea $f$ radial. Demuestre que $\Phi*f$ es radial.

4. **Tipo Examen General.** Sea $f\in C_c^\infty(\mathbb R^3)$ y
   $u=\Phi*f$. Demuestre que
   $$
   u(x)=O(|x|^{-1})
   $$
   cuando $|x|\to\infty$, e identifique el primer término asintótico en función de
   $\int_{\mathbb R^3}f$.

# 3.2.2 Propiedad del promedio

#### **Transcripción literal de las notas**

Sea $\Omega\subset\mathbb R^n$ abierto y $u\in C^2(\Omega)$.

$u$ es armónica en $\Omega$ si y sólo si satisface la propiedad del promedio:

$$
u(x)
=
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u(y)\,dS_y
=
\frac{1}{|B_r(x)|}\int_{B_r(x)}u(y)\,dy
$$

para toda bola $B_r(x)\subset\Omega$.

## **Definición 3.2.2 (Promedio superficial y promedio volumétrico).**

Sea $u$ integrable sobre $\partial B_r(x)$ y sobre $B_r(x)$. Se definen

$$
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS
=
\frac{1}{|\partial B_r|}
\int_{\partial B_r(x)}u\,dS
$$

y

$$
\frac{1}{|B_r(x)|}\int_{B_r(x)}u\,dy
=
\frac{1}{|B_r|}
\int_{B_r(x)}u\,dy.
$$

## **Teorema 3.2.3 (Propiedad del promedio para funciones armónicas).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C^2(\Omega)$. Si

$$
\Delta u=0
\qquad\text{en }\Omega,
$$

entonces para toda bola cerrada $\overline{B_r(x)}\subset\Omega$,

$$
u(x)
=
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u(y)\,dS_y
=
\frac{1}{|B_r(x)|}\int_{B_r(x)}u(y)\,dy.
$$

### Demostración añadida: promedio sobre esferas

Fijemos $x\in\Omega$ y definamos

$$
M(r)
=
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS
=
\frac{1}{\omega_n}
\int_{\partial B_1(0)}u(x+r\theta)\,dS_\theta.
$$

Derivando bajo el signo integral,

$$
M'(r)
=
\frac{1}{\omega_n}
\int_{\partial B_1(0)}
\nabla u(x+r\theta)\cdot\theta\,dS_\theta.
$$

Al cambiar variables $y=x+r\theta$,

$$
M'(r)
=
\frac{1}{\omega_n r^{n-1}}
\int_{\partial B_r(x)}
\frac{\partial u}{\partial\nu}\,dS.
$$

Por el teorema de la divergencia,

$$
\int_{\partial B_r(x)}
\frac{\partial u}{\partial\nu}\,dS
=
\int_{B_r(x)}\Delta u\,dy
=
0.
$$

Así, $M'(r)=0$, por lo que $M$ es constante. Como $u$ es continua,

$$
\lim_{r\downarrow0}M(r)=u(x).
$$

Por tanto,

$$
M(r)=u(x).
$$

### Demostración añadida: promedio sobre bolas

Usando coordenadas polares alrededor de $x$,

$$
\begin{aligned}
\int_{B_r(x)}u(y)\,dy
&=
\int_0^r
\int_{\partial B_\rho(x)}u(y)\,dS_y\,d\rho\\
&=
\int_0^r
|\partial B_\rho|u(x)\,d\rho\\
&=
|B_r|u(x).
\end{aligned}
$$

Dividiendo entre $|B_r|$ se obtiene la fórmula volumétrica.

$\square$

## **Teorema 3.2.4 (Recíproco de la propiedad del promedio).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C^2(\Omega)$. Supongamos que para todo
$x\in\Omega$ y todo $r>0$ con $\overline{B_r(x)}\subset\Omega$ se cumple

$$
u(x)
=
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS.
$$

Entonces $u$ es armónica en $\Omega$.

### Demostración añadida

Para $x\in\Omega$ fijo, la expansión de Taylor da

$$
u(x+r\theta)
=
u(x)
+
r\nabla u(x)\cdot\theta
+
\frac{r^2}{2}\theta^TD^2u(x)\theta
+
o(r^2)
$$

uniformemente para $\theta\in\partial B_1$.

Promediando sobre la esfera y usando

$$
\frac{1}{|\partial B_1|}\int_{\partial B_1}\theta_i\,dS=0,
$$

y

$$
\frac{1}{|\partial B_1|}\int_{\partial B_1}\theta_i\theta_j\,dS
=
\frac{\delta_{ij}}{n},
$$

obtenemos

$$
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS
=
u(x)+\frac{r^2}{2n}\Delta u(x)+o(r^2).
$$

Por hipótesis, el miembro izquierdo es $u(x)$. Dividiendo entre $r^2$ y haciendo
$r\downarrow0$,

$$
\Delta u(x)=0.
$$

Como $x$ es arbitrario, $u$ es armónica en $\Omega$.

$\square$

### Ejercicios — Sección 3.2.2

1. Verifique directamente la propiedad del promedio para
   $$
   u(x,y)=x^2-y^2
   $$
   en discos centrados en el origen.

2. Demuestre las identidades
   $$
   \frac{1}{|\partial B_1|}\int_{\partial B_1}\theta_i\,dS=0,
   \qquad
   \frac{1}{|\partial B_1|}\int_{\partial B_1}\theta_i\theta_j\,dS=\frac{\delta_{ij}}{n}.
   $$

3. Demuestre el recíproco usando promedios sobre bolas en lugar de promedios sobre esferas.

4. **Tipo Examen General.** Sea $u\in C(\Omega)$ y suponga que satisface la propiedad del
   promedio sobre todas las bolas compactamente contenidas en $\Omega$. Demuestre que
   $u\in C^\infty(\Omega)$ y que $\Delta u=0$. Puede usar una regularización por convolución,
   pero debe justificar cada paso.

# 3.2.3 Subarmonicidad

#### **Transcripción literal de las notas**

Sea $\Omega\subset\mathbb R^n$ abierto y $u\in C^2(\Omega)$.

Se dice que $u$ es subarmónica en $\Omega$ si

$$
u(x)
\le
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u(y)\,dS_y
$$

y también

$$
u(x)
\le
\frac{1}{|B_r(x)|}\int_{B_r(x)}u(y)\,dy
$$

para toda bola $B_r(x)\subset\Omega$.

Además, para $u\in C^2(\Omega)$:

$$
u\text{ es subarmónica}
\Longleftrightarrow
\Delta u\ge0
\quad\text{en }\Omega.
$$

## **Definición 3.2.5 (Función subarmónica: formulación por promedios).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C(\Omega)$. Se dice que $u$ es
**subarmónica en $\Omega$** si para toda bola cerrada $\overline{B_r(x)}\subset\Omega$,

$$
u(x)
\le
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS.
$$

## **Definición 3.2.6 (Función superarmónica).**

Se dice que $u$ es **superarmónica** si $-u$ es subarmónica. Equivalentemente,

$$
u(x)
\ge
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS.
$$

## **Teorema 3.2.7 (Criterio diferencial de subarmonicidad).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C^2(\Omega)$. Entonces son equivalentes:

1. $u$ es subarmónica en el sentido de la Definición 3.2.5;
2. para toda bola cerrada $\overline{B_r(x)}\subset\Omega$,
   $$
   u(x)\le\frac{1}{|B_r(x)|}\int_{B_r(x)}u(y)\,dy;
   $$
3. se cumple
   $$
   \Delta u\ge0
   \qquad\text{en }\Omega.
   $$

### Demostración añadida de $3\Rightarrow1$

Para $M(r)=\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS$, el cálculo del Teorema 3.2.3 da

$$
M'(r)
=
\frac{1}{\omega_n r^{n-1}}
\int_{B_r(x)}\Delta u\,dy.
$$

Si $\Delta u\ge0$, entonces $M'(r)\ge0$. Por tanto,

$$
M(r)\ge\lim_{\rho\downarrow0}M(\rho)=u(x).
$$

### Demostración añadida de $1\Rightarrow3$

Por la expansión de Taylor,

$$
\frac{1}{|\partial B_r(x)|}\int_{\partial B_r(x)}u\,dS-u(x)
=
\frac{r^2}{2n}\Delta u(x)+o(r^2).
$$

El miembro izquierdo es no negativo. Dividiendo entre $r^2$ y pasando al límite,

$$
\Delta u(x)\ge0.
$$

La equivalencia con el promedio volumétrico se obtiene integrando los promedios superficiales en
el radio.

$\square$

### Ejemplo 3.2.1

La función

$$
u(x)=|x|^2
$$

satisface

$$
\Delta u=2n>0.
$$

Por tanto, es estrictamente subarmónica.

### Ejemplo 3.2.2

Toda función armónica es simultáneamente subarmónica y superarmónica.

### Contraejemplo 3.2.1

La desigualdad del promedio en un solo radio no basta para concluir subarmonicidad. La definición
exige que se cumpla para todas las bolas suficientemente pequeñas compactamente contenidas en el
dominio.

### Ejercicios — Sección 3.2.3

1. Determine para qué valores de $\alpha\in\mathbb R$ la función
   $$
   u(x)=|x|^\alpha
   $$
   es subarmónica en $\mathbb R^n\setminus\{0\}$.

2. Demuestre que el máximo de dos funciones subarmónicas continuas es subarmónico.

3. Sea $\phi:\mathbb R\to\mathbb R$ convexa y no decreciente, y sea $u$ subarmónica de clase
   $C^2$. Demuestre que $\phi\circ u$ es subarmónica.

4. **Tipo Examen General.** Sea $\Omega\subset\mathbb R^n$ abierto y sea
   $u\in C^2(\Omega)$ tal que
   $$
   \Delta u\ge c>0.
   $$
   Demuestre una desigualdad cuantitativa entre $u(x)$ y su promedio sobre
   $\partial B_r(x)$, válida para toda bola compactamente contenida en $\Omega$.

5. **Tipo Examen General.** Sea $u\in C^2(\Omega)$ y suponga que para cada
   $\overline{B_r(x)}\subset\Omega$,
   $$
   u(x)\le\frac{1}{|B_r(x)|}\int_{B_r(x)}u.
   $$
   Demuestre directamente que $\Delta u(x)\ge0$ usando una expansión de Taylor con resto.

# Control de cobertura y estado del capítulo

## Contenido cubierto en este notebook

- representación de una solución de Poisson mediante la solución fundamental;
- propiedad del promedio sobre esferas;
- propiedad del promedio sobre bolas;
- recíproco de la propiedad del promedio;
- definición de subarmonicidad y superarmonicidad;
- equivalencia entre subarmonicidad y la desigualdad $\Delta u\ge0$ para funciones $C^2$;
- visualizaciones numéricas de los promedios.

## Contenido añadido para cerrar huecos

- hipótesis de soporte compacto en la representación por convolución;
- formulación distribucional y regularidad interior;
- demostraciones completas de las propiedades del promedio;
- demostración del recíproco mediante Taylor;
- equivalencia entre las formulaciones superficial, volumétrica y diferencial.

## Contenido pendiente inmediato

La página 7 de las notas comienza con:

- principio débil del máximo;
- principio de comparación;
- super y subsoluciones;
- operadores con término de orden cero.

El siguiente notebook será:

$$
\texttt{03.3.01\_Principios\_del\_maximo\_unicidad\_estabilidad.ipynb}.
$$